# Breast Cancer Classification: Reproducible ML Benchmark

A compact, end-to-end machine learning workflow using the scikit-learn breast cancer dataset. The notebook emphasizes reproducibility, leakage-safe preprocessing, model comparison, and interpretable evaluation.

## Objectives

- Establish a reproducible train/test split.
- Compare a scaled linear baseline with a tree ensemble.
- Report accuracy, precision, recall, F1, and ROC AUC.
- Identify the most influential features for the best model.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

print(f'Dataset shape: {X.shape}')
print(f'Class distribution: {y.value_counts().sort_index().to_dict()}')
print(f'Missing values: {int(X.isna().sum().sum())}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

models = {
    'Logistic Regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    )
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    predictions[name] = pred
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
display(results_df.round(4))

In [ ]:
best_name = results_df.iloc[0]['model']
print(f'Best model by ROC AUC: {best_name}')
print()
print(classification_report(y_test, predictions[best_name], target_names=data.target_names))

In [ ]:
if best_name == 'Random Forest':
    importance = models[best_name].feature_importances_
else:
    importance = np.abs(models[best_name].named_steps['model'].coef_[0])

top_features = (
    pd.Series(importance, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)
print('Top 10 influential features:')
display(top_features.to_frame('importance').round(4))
print('Pipeline completed successfully.')

## Conclusion

This benchmark provides a reproducible baseline for binary classification. In a production setting, the next steps would include external validation, threshold selection based on business costs, calibration, and monitoring for data drift.